In [1]:
import os
import numpy.random as rd
import matplotlib.pyplot as plt
from functools import partial
import numpy as np
import time
import pyvisa as pvisa
import qcodes as qc
from qcodes.instrument_drivers.yehlab_driver.QD import QD
import qcodes.instrument_drivers.nplab_drivers as npd
import qcodes.instrument_drivers.yehlab_driver as yd
from qcodes.instrument_drivers.stanford_research.SR830 import SR830
from qcodes.instrument_drivers.Keithley.Keithley_2636B import *












In [2]:
lockin1=SR830('lockin1','GPIB0::8::INSTR') # SR830 measurement



Connected to: Stanford_Research_Systems SR830 (serial:s/n92421, firmware:ver1.07) in 0.65s


C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument lockin1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument lockin1 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


In [3]:
lockin2=SR830('lockin2','GPIB0::3::INSTR') # SR830 AC

Connected to: Stanford_Research_Systems SR830 (serial:s/n32765, firmware:ver1.07) in 0.17s


C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch1_databuffer on instrument lockin2 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(
C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\qcodes\instrument\instrument_base.py:143: QCoDeSDeprecationWarning: Parameter ch2_databuffer on instrument lockin2 does not correctly pass kwargs to its baseclass. A Parameter class must take `**kwargs` and forward them to its baseclass.
  warnings.warn(


In [4]:
k2636B = Keithley2600('k2636B','USB0::0x05E6::0x2636::4353546::INSTR')

Connected to: Keithley Instruments Inc. 2636B (serial:4353546, firmware:3.3.5) in 0.16s


In [5]:
k2636B.smua.mode()

'voltage'

In [6]:
k2636B.smub.mode()

'voltage'

In [28]:
k2636B.smub.volt(0)

In [8]:
lockin1.write("FMOD 1")     # 内部参考模式
lockin1.write("FREQ 311")   # 参考频率 (Hz)
lockin1.write("SLVL 0.2")   # 输出电压振幅 (V)
lockin1.write("ISRC 0")     # 模式（0 = Voltage, 1 = Current）

In [9]:
lockin1.write("FREQ 311")   # 参考频率 (Hz)

In [10]:
lockin2.input_coupling.set("AC")  
lockin2.write("FMOD 1") 	  # lockin参考信號為外部输入（REF IN）
lockin2.write("IGND 0")  	  # 設定輸入不接地（FLOAT）
lockin2.harmonic.set(1)		  # 測量 1 倍頻成分

In [11]:
lockin2.write('AGAN')		  #自動檔位 sensitivity

In [188]:
lockin1.write("SLVL 0.04")   # 输出电压振幅 (V)

In [193]:
volt_val = np.linspace(0.04, 1, 100)
k2636B.smub.volt(-20)
k2636B.smub.output(1)
lockin1.write("SLVL 1") 


In [190]:
lockin2.write('AGAN')		  #自動檔位 sensitivity

In [194]:
lockin1.write("SLVL 0.004") 

In [195]:
time.sleep(1)  # 稳定
data1 , plot1=npd.single_param_sweep(lockin1.amplitude, volt_val, 0.5, lockin2.R, lockin2.P, lockin2.X, lockin2.Y, DataName="nolight Gate=10-test4")
time.sleep(1)
k2636B.smub.volt(0)
k2636B.smub.output(0)
lockin1.write("SLVL 0.004") 

Started at 2025-08-13 00:24:30
DataSet:
   location = 'data/2025-08-13/#013_nolight Gate=10-test4_00-24-28'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (100,)
   Measured | lockin2_R             | R            | (100,)
   Measured | lockin2_P             | P            | (100,)
   Measured | lockin2_X             | X            | (100,)
   Measured | lockin2_Y             | Y            | (100,)
Finished at 2025-08-13 00:25:35


In [197]:
x = 20
volt_val = np.linspace(0.004, 0.7, 100)
while x >= -20:
    if x == -20:
        time.sleep(1)
        k2636B.smub.volt(0)
        k2636B.smub.output(0)
        lockin1.write("SLVL 0.004") 
    else:
        time.sleep(1)
        k2636B.smub.volt(0)
        k2636B.smub.output(0)
        lockin1.write("SLVL 0.004") 
        time.sleep(1)
        k2636B.smub.volt(x)
        time.sleep(3)
        k2636B.smub.output(1)
        time.sleep(3)  # 稳定
        data1 , plot1=npd.single_param_sweep(lockin1.amplitude, volt_val, 0.5, lockin2.R, lockin2.P, lockin2.X, lockin2.Y, DataName="Mono-Gate"+str(x))
        time.sleep(1)
    x=x-1

Started at 2025-08-13 01:22:32
DataSet:
   location = 'data/2025-08-13/#054_Mono-Gate20_01-22-30'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (100,)
   Measured | lockin2_R             | R            | (100,)
   Measured | lockin2_P             | P            | (100,)
   Measured | lockin2_X             | X            | (100,)
   Measured | lockin2_Y             | Y            | (100,)
Finished at 2025-08-13 01:23:37
Started at 2025-08-13 01:23:49
DataSet:
   location = 'data/2025-08-13/#055_Mono-Gate19_01-23-46'
   <Type>   | <array_id>            | <array.name> | <array.shape>
   Setpoint | lockin1_amplitude_set | amplitude    | (100,)
   Measured | lockin2_R             | R            | (100,)
   Measured | lockin2_P             | P            | (100,)
   Measured | lockin2_X             | X            | (100,)
   Measured | lockin2_Y             | Y            | (100,)
Finished at 2025-08-13 01:24:53
Starte

In [182]:
time.sleep(1)
k2636B.smub.volt(0)
k2636B.smub.output(0)
lockin1.write("SLVL 0.004") 